<a href="https://colab.research.google.com/github/nitka975/MachinelLearning/blob/main/DeepFake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/KsiuTretyakova/first-order-model.git
%cd first-order-model
!pip install ffmpeg-python
!pip install imageio
!pip install imageio-ffmpeg

Cloning into 'first-order-model'...
remote: Enumerating objects: 391, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 391 (delta 0), reused 0 (delta 0), pack-reused 388 (from 2)
Receiving objects: 100% (391/391), 72.19 MiB | 38.11 MiB/s, done.
Resolving deltas: 100% (200/200), done.
/content/first-order-model/first-order-model


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
from demo import load_checkpoints
root_path = '/content/drive/My Drive/First Order Model/'
generator, kp_detector = load_checkpoints(config_path=root_path+'vox-256.yaml',
                                          checkpoint_path=root_path+'vox-cpk.pth.tar')

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [ ]:
face_path= 'Nikita.jpg'  #@param {type:"string"}
drive_path= 'driver.mp4'  #@param {type:"string"}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

import warnings
warnings.filterwarnings("ignore")

def display(source, driving, generated=None):
  fig = plt.figure(figsize=(8 + 4 * (generated is not None), 6)) # 12x6

  ims = []
  for i in range(len(driving)):
    cols = [source]
    cols.append(driving[i])
    if generated is not None:
      cols.append(generated[i])
    im = plt.imshow(np.concatenate(cols, axis=1), animated=True)
    plt.axis('off')
    ims.append([im])

  ani = animation.ArtistAnimation(fig, ims, interval=50, repeat_delay=1000)
  plt.close()
  return ani

In [ ]:
import imageio
from demo import make_animation
from skimage import img_as_ubyte
from skimage.transform import resize
from IPython.display import HTML

source_image = imageio.imread(root_path + face_path)
reader = imageio.get_reader(root_path + drive_path)

fps = reader.get_meta_data()['fps']

driving_video = []
try:
  for im in reader:
    driving_video.append(im)
except RuntimeError:
  pass
except Exception as e:
  print(f"ERROR: {e}")
reader.close()

source_image = resize(source_image, (256, 256))[..., :3]
driving_video = [resize(frame, (256, 256))[..., :3] for frame in driving_video]


predictions = make_animation(
    source_image,
    driving_video,
    generator,
    kp_detector,
    relative=True,
    adapt_movement_scale=True)

imageio.mimsave('../output.mp4', [img_as_ubyte(frame) for frame in predictions ])

HTML(display(source_image, driving_video, predictions).to_html5_video())

  0%|          | 0/852 [00:00<?, ?it/s]